In [ ]:
# ============================================
# Imports & Device
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import numpy as np
from tqdm import tqdm
from efficientnet_pytorch import EfficientNet
import os
import random

In [ ]:
# ============================================
# Configuration
# ============================================

train_dir = "../data/dataset/train"
val_dir   = "../data/dataset/val"

batch_size = 32
num_workers = 4
mixup_alpha = 0.2
label_smoothing = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# ============================================
# Data Augmentation (Optimale)
# ============================================

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(300, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.05),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [ ]:
# ============================================
# Datasets & Loaders
# ============================================

train_ds = datasets.ImageFolder(train_dir, transform=train_transforms)
val_ds   = datasets.ImageFolder(val_dir,   transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

num_classes = len(train_ds.classes)
num_classes

In [ ]:
# ============================================
# Mixup
# ============================================

def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, 1.0

    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# ============================================
# EfficientNet-B3
# ============================================

model = EfficientNet.from_pretrained("efficientnet-b3")
model._fc = nn.Linear(model._fc.in_features, num_classes)
model = model.to(device)

In [ ]:
# ============================================
# Train & Validation Functions
# ============================================

def train_one_epoch(model, loader, optimizer, criterion, mixup_alpha=0.2):
    model.train()
    total, correct, total_loss = 0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Mixup
        x, y_a, y_b, lam = mixup_data(x, y, alpha=mixup_alpha)

        optimizer.zero_grad()
        preds = model(x)
        loss = mixup_criterion(criterion, preds, y_a, y_b, lam)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        _, predicted = preds.max(1)
        correct += (lam * predicted.eq(y_a).sum().item()
                    + (1 - lam) * predicted.eq(y_b).sum().item())
        total += x.size(0)

    return total_loss / total, correct / total


def validate(model, loader, criterion):
    model.eval()
    total, correct, total_loss = 0, 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)
            loss = criterion(preds, y)

            total_loss += loss.item() * x.size(0)
            _, predicted = preds.max(1)
            correct += predicted.eq(y).sum().item()
            total += x.size(0)

    return total_loss / total, correct / total

In [ ]:
# ============================================
# Phase 1 — Feature Extraction
# ============================================

for param in model.parameters():
    param.requires_grad = False

for param in model._fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
optimizer = optim.Adam(model._fc.parameters(), lr=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=5)

EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate(model, val_loader, criterion)
    scheduler.step()

    print(f"Phase 1 — Epoch {epoch+1}/{EPOCHS} | "
          f"train acc {train_acc:.3f} | val acc {val_acc:.3f}")

In [ ]:
# ============================================
# Phase 2 — Fine-tuning complet
# ============================================

for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=15)

best_acc = 0
EPOCHS = 15

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate(model, val_loader, criterion)
    scheduler.step()

    print(f"Phase 2 — Epoch {epoch+1}/{EPOCHS} | "
          f"train acc {train_acc:.3f} | val acc {val_acc:.3f}")

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "efficientnet_b3_best.pth")
        print("  ✓ Best model saved")

In [ ]:
# ============================================
# Load Best Model
# ============================================

model.load_state_dict(torch.load("efficientnet_b3_best.pth"))
model.eval()
print("Best model loaded.")